In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AdamW
from torch.nn.utils.rnn import pad_sequence
from torch.nn.functional import pad
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer

In [ ]:
from datasets import load_dataset
eli5 = load_dataset("eli5", split="train_asks[:5000]")
eli5 = eli5.train_test_split(test_size=0.2)

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
eli5 = eli5.flatten()

def preprocess_function(examples):
    return tokenizer([" ".join(x) for x in examples["answers.text"]])

In [ ]:
tokenized_eli5 = eli5.map(
    preprocess_function,
    batched=True,
    num_proc=1,
    remove_columns=eli5["train"].column_names,
)

In [ ]:
block_size = 128


def group_texts(examples):
    # Concatenate all texts.
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    # We drop the small remainder, we could add padding if the model supported it instead of this drop, you can
    # customize this part to your needs.
    if total_length >= block_size:
        total_length = (total_length // block_size) * block_size
    # Split by chunks of block_size.
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result
lm_dataset = tokenized_eli5.map(group_texts, batched=True, num_proc=1)

In [ ]:
class Dataset(Dataset):
    def __init__(self,lm_dataset):
        self.lm_dataset=lm_dataset
        self.input_ids=[torch.tensor(x) for x in self.lm_dataset['train']['input_ids']]
        self.labels=[torch.tensor(x) for x in self.lm_dataset['train']['labels']]
        
    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        # Tokenize and encode the sequence
        return self.input_ids[idx], self.labels[idx]

In [ ]:
dataset=Dataset(lm_dataset)
len(dataset[67][0])

In [ ]:
model = AutoModelForCausalLM.from_pretrained("distilgpt2")

dataloader = DataLoader(dataset, batch_size=8,shuffle=True)

# Set up optimizer and loss function
optimizer = AdamW(model.parameters(), lr=5e-4)
criterion = nn.CrossEntropyLoss()

# Training loop
num_epochs = 100
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

dataloader

In [ ]:
from tqdm import tqdm
import gc

In [ ]:
torch.cuda.empty_cache()
gc.collect()

In [ ]:
for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0

    for inp, lab in tqdm(dataloader):
        inputs = inp.to(device)
        labels = lab.to(device)

        # Forward pass
        outputs = model(inputs, labels=labels)
        loss = outputs.loss

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(dataloader)
    print(f'Epoch {epoch + 1}/{num_epochs}, Loss: {average_loss}')


In [ ]:
torch.save(model, 'example_distiledGPT.pth')